# 实验2.1 昇腾嵌入式AI开发基础探究实验

> **课程**：AI 系统导论  
> **实验平台**：GitCode CANNLab 云沙箱（昇腾 Atlas NPU · Ascend 910B3）  
> **CANN 版本**：9.0 ｜ **Python**：3.11.4 ｜ **内核**：ARM  

---

## 实验概述

本实验基于已预先调通并搭建好的云沙箱基础环境（已完成 CANN 架构版本更新与运行环境配置）开展，聚焦**昇腾嵌入式 AI 开发基础**中的关键问题点进行设计。与传统的本地环境搭建实验不同，本实验不再要求学生在环境安装与版本适配上投入大量时间，而是让学生在**已配置完成的环境中直接进入核心实践环节**，围绕昇腾嵌入式 AI 开发的基础知识与核心能力展开探究。

在已配置完成的环境中，学生重点实践以下内容：

1. **运用工具查询开发硬件信息**——监控 CPU 运行状态及 AI 处理器负载
2. **运行官方配套示例程序**——验证环境可用性
3. **通过针对性问题探究**——掌握昇腾端侧开发、部署与调试的基础流程和方法

云沙箱环境由 GitCode Notebook 提供，底层搭载昇腾 Atlas 系列 NPU（本实验实际分配到的芯片为 **Ascend 910B3**），并已预装好对应版本的昇腾 AI 处理器配套软件栈 **CANN**（Compute Architecture for Neural Networks）。沙箱环境开箱即用，但不允许用户自行修改 CANN 版本，因此本实验跳过版本切换环节，直接进入调试与验证实践。

### 实验目标

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">目标</th>
<th style="text-align: left;">对应章节</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">学会使用系统命令获取硬件、OS、CANN、Python 等环境信息</td>
<td style="text-align: left;">一</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">能够编写并运行 Python 基础程序、多核 CPU 并行程序、NPU 加速程序</td>
<td style="text-align: left;">二</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">通过动手练习加深对昇腾嵌入式 AI 开发的理解</td>
<td style="text-align: left;">三</td>
</tr>
</table>


---

# 一、开发环境信息获取

在本节中，我们将产生合适的命令，直接在 Jupyter 环境下运行，获取以下信息：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">任务</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">系统硬件信息</td>
<td style="text-align: left;">CPU 信息、NPU 信息、内存信息等</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">操作系统信息</td>
<td style="text-align: left;">获取操作系统类型及版本等</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">Python 信息</td>
<td style="text-align: left;">获取 Python 版本号等</td>
</tr>
</table>

> 📝 **学习目标**：通过本节练习，掌握在昇腾 NPU 环境下查询硬件与软件信息的方法。


## 2.1 系统硬件信息

### 2.1.1 CPU 信息

`lscpu` 命令可以显示 CPU 架构、核心数、主频等详细信息。


**代码功能说明：**

使用 `lscpu` 命令获取 CPU 的详细信息。

**`lscpu` 命令原理：**

- `lscpu` 从 `/sys/devices/system/cpu/` 和 `/proc/cpuinfo` 中读取 CPU 信息并格式化输出
- 在本环境中，你会看到：
  - `Architecture: aarch64`（ARM 架构，昇腾服务器采用 ARM CPU）
  - `CPU(s): 16`（16 个逻辑核心）
  - `Model name`（CPU 型号）
  - `CPU max MHz`（最大主频）

> **知识点**：`!命令` 是 Jupyter 的魔法语法，可以在 Notebook 中直接运行 Shell 命令。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取 CPU 详细信息
print("=" * 60)
print("CPU 信息 (lscpu)")
print("=" * 60)
!lscpu


**代码功能说明：**

用多种方式获取 CPU 核心数，并对比结果是否一致。

**三种方式对比：**

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方式</th>
<th style="text-align: left;">原理</th>
<th style="text-align: left;">返回值</th>
</tr>
<tr>
<td style="text-align: left;"><code>os.cpu_count()</code></td>
<td style="text-align: left;">Python 标准库，读取系统信息</td>
<td style="text-align: left;">逻辑核心数</td>
</tr>
<tr>
<td style="text-align: left;"><code>multiprocessing.cpu_count()</code></td>
<td style="text-align: left;">multiprocessing 模块，用于确定并行进程数</td>
<td style="text-align: left;">逻辑核心数</td>
</tr>
<tr>
<td style="text-align: left;"><code>nproc</code></td>
<td style="text-align: left;">Shell 命令，读取 <code>/proc/cpuinfo</code></td>
<td style="text-align: left;">逻辑核心数</td>
</tr>
</table>

> 三种方式结果通常相同。如果不同，可能是因为容器或虚拟环境限制了可用核心数。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取 CPU 核心数（多种方式对比）
import os
import multiprocessing

print("CPU 核心数获取方式对比：")
print(f"  os.cpu_count()          = {os.cpu_count()}")
print(f"  multiprocessing.cpu_count() = {multiprocessing.cpu_count()}")
!echo "  nproc                   = $(nproc)"
!echo "  /proc/cpuinfo 核心数     = $(grep -c processor /proc/cpuinfo)"


**代码功能说明：**

读取 `/proc/cpuinfo` 文件查看 CPU 详细信息。

**`/proc` 虚拟文件系统原理：**

- `/proc` 是 Linux 的**虚拟文件系统**，内容存在于内存中，不占磁盘空间
- `/proc/cpuinfo` 包含每个 CPU 核心的详细信息：型号、主频、缓存大小、标志位等
- 与 `lscpu` 相比，`/proc/cpuinfo` 信息更原始、更详细
- `head -30` 表示只显示前 30 行（避免输出过长）

> **知识点**：Linux 中很多系统信息都可以通过读取 `/proc` 下的文件获取，如 `/proc/meminfo`（内存）、`/proc/version`（内核版本）等。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 查看 /proc/cpuinfo 中的详细信息（前30行）
print("/proc/cpuinfo（前30行）：")
!cat /proc/cpuinfo | head -30


### 2.1.2 NPU（昇腾 AI 处理器）信息

`npu-smi` 是昇腾 NPU 的系统管理工具（类似 NVIDIA 的 `nvidia-smi`），可以查看 NPU 的型号、状态、利用率、显存等信息。

常用命令：
- `npu-smi info`：查看 NPU 概览信息
- `npu-smi info -l`：查看 NPU 设备列表
- `npu-smi info -t board`：查看板卡详细信息
- `npu-smi info -t usages -i 0`：查看指定设备的资源使用情况


**代码功能说明：**

使用 `npu-smi info` 命令获取 NPU（昇腾 AI 处理器）的概览信息。

**`npu-smi` 命令原理：**

- `npu-smi` 是昇腾 NPU 的系统管理工具，功能类似 NVIDIA GPU 的 `nvidia-smi`
- `npu-smi info` 显示所有 NPU 设备的概览信息，包括：
  - **NPU 编号**和**芯片编号**
  - **Name**：NPU 型号（如 910B3）
  - **Health**：健康状态（OK 表示正常）
  - **Power(W)**：当前功耗
  - **Temp(C)**：当前温度
  - **AICore(%)**：AI 核心利用率
  - **Memory-Usage**：显存使用情况

> **知识点**：`npu-smi` 是 CANN 工具链的一部分，安装 CANN 后即可使用。它是监控 NPU 状态最重要的命令。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取 NPU 概览信息
print("=" * 60)
print("NPU 信息 (npu-smi info)")
print("=" * 60)
!npu-smi info


**代码功能说明：**

使用 `npu-smi info -l` 查看 NPU 设备的**拓扑信息**（设备列表）。

**`-l` 参数原理：**

- `-l` 显示所有 NPU 设备的拓扑结构
- 在多卡服务器上，可以看到多块 NPU 之间的连接关系（如 HCCS 互联）
- 本实验环境通常只有 1 块 NPU，因此拓扑信息较简单

> **对比**：`npu-smi info` 显示运行状态，`npu-smi info -l` 显示硬件拓扑结构。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取 NPU 设备列表
print("NPU 设备列表：")
!npu-smi info -l


**代码功能说明：**

获取 NPU 板卡的**详细信息**（型号、序列号、固件版本等）。

**原理与关键点：**

- `npu-smi info -t board` 需要 `-i <Card ID>` 参数指定哪块卡
- 不同环境的 Card ID 可能不同（不一定是 0）
- 因此先用 `subprocess.run()` 运行 `npu-smi info`，从输出中**自动解析** Card ID
- 再用解析到的 Card ID 查询板卡详细信息和资源使用情况

> **为什么用 Python 而非直接用 `!` 命令？** 因为需要先解析输出再决定参数，有逻辑依赖关系，用 Python 的 `subprocess` 模块更灵活。


In [ ]:
# 获取 NPU 板卡详细信息（自动解析 Card ID）
import subprocess

# 先从 npu-smi info 输出中解析 Card ID
result = subprocess.run(["npu-smi", "info"], capture_output=True, text=True)
print("npu-smi info 输出：")
print(result.stdout)

# 若命令失败或无输出，说明原因
if result.returncode != 0 or not result.stdout.strip():
    print("\n无法获取 NPU 信息，可能原因：")
    print("  1) 未安装 CANN 工具链（npu-smi 不可用）")
    print("  2) 当前用户无 NPU 访问权限")
    print("  3) 未挂载 NPU 设备")
    if result.stderr.strip():
        print(f"  错误信息：{result.stderr.strip()}")

# 尝试解析 Card ID
# 按 '|' 分列：Card 行第一列含 "<ID> <Name>" 两个 token，Chip 行仅含 "<ID>"
card_ids = set()
for line in result.stdout.split("\n"):
    if not line.startswith("|"):
        continue
    fields = [f.strip() for f in line.split("|")]
    # fields[1] 是第一列：Card 行为 "<ID> <Name>"，Chip 行仅为 "<ID>"
    if len(fields) >= 2 and fields[1]:
        col1 = fields[1].split()
        if len(col1) >= 2 and col1[0].isdigit():
            card_ids.add(int(col1[0]))

if card_ids:
    for cid in sorted(card_ids):
        print(f"\n--- Card ID {cid} 板卡信息 ---")
        r = subprocess.run(["npu-smi", "info", "-t", "board", "-i", str(cid)],
                           capture_output=True, text=True)
        if r.stdout.strip():
            print(r.stdout)
        if r.stderr.strip():
            print(f"(board 信息不可用: {r.stderr.strip()})")

        # 尝试获取资源使用情况
        print(f"\n--- Card ID {cid} 资源使用 ---")
        r2 = subprocess.run(["npu-smi", "info", "-t", "usages", "-i", str(cid)],
                            capture_output=True, text=True)
        if r2.stdout.strip():
            print(r2.stdout)
        if r2.stderr.strip():
            print(f"(usages 信息不可用: {r2.stderr.strip()})")
else:
    print("未能从 npu-smi info 解析出 Card ID（请检查上方输出是否包含 NPU 设备行）")


**代码功能说明：**

获取 NPU 的**温度、功率、显存**等运行时监控信息。

**各监控项含义：**

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">类型</th>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">温度</td>
<td style="text-align: left;"><code>-t temp</code></td>
<td style="text-align: left;">NPU 芯片当前温度，过高会自动降频保护</td>
</tr>
<tr>
<td style="text-align: left;">功率</td>
<td style="text-align: left;"><code>-t power</code></td>
<td style="text-align: left;">NPU 当前功耗（瓦），反映负载程度</td>
</tr>
<tr>
<td style="text-align: left;">显存</td>
<td style="text-align: left;"><code>-t memory</code></td>
<td style="text-align: left;">NPU 显存使用情况，类似 GPU 显存</td>
</tr>
</table>

> **实际应用**：在训练大模型时，需要监控温度和功率，防止 NPU 过热或功耗超限。


In [ ]:
# 获取 NPU 温度和功率信息（自动解析 Card ID）
import subprocess

result = subprocess.run(["npu-smi", "info"], capture_output=True, text=True)
if result.returncode != 0 or not result.stdout.strip():
    print("无法获取 NPU 信息，可能原因：未安装 CANN / 无 NPU 访问权限 / 未挂载 NPU")
card_ids = set()
for line in result.stdout.split("\n"):
    if not line.startswith("|"):
        continue
    fields = [f.strip() for f in line.split("|")]
    # fields[1] 是第一列：Card 行为 "<ID> <Name>"，Chip 行仅为 "<ID>"
    if len(fields) >= 2 and fields[1]:
        col1 = fields[1].split()
        if len(col1) >= 2 and col1[0].isdigit():
            card_ids.add(int(col1[0]))

for cid in sorted(card_ids):
    for t in ["temp", "power", "memory"]:
        r = subprocess.run(["npu-smi", "info", "-t", t, "-i", str(cid)],
                           capture_output=True, text=True)
        if r.stdout.strip():
            print(f"--- Card {cid} {t} ---")
            print(r.stdout)


### 2.1.3 内存信息

使用 `free` 和 `/proc/meminfo` 查看系统内存使用情况。


**代码功能说明：**

使用 `free -h` 和 `/proc/meminfo` 查看系统**内存使用情况**。

**命令原理：**

- `free -h`：以人类可读格式（GB/MB）显示内存总量、已用、可用、缓存和交换分区
- `/proc/meminfo`：更详细的内存信息，每行一个指标
  - `MemTotal`：总内存
  - `MemAvailable`：可用内存（含可回收缓存）
  - `SwapTotal`：交换分区总量

> **知识点**：Linux 会把空闲内存用作磁盘缓存（Cache/Buffers），这不是"浪费"，而是加速文件读取。`MemAvailable` 才是真正可用的内存。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取内存信息
print("=" * 60)
print("内存信息 (free -h)")
print("=" * 60)
!free -h

print()
print("/proc/meminfo（前10行）：")
!cat /proc/meminfo | head -10


### 2.1.4 磁盘信息

> 📖：在实际开发中，了解磁盘空间有助于判断模型存储和数据加载是否充足。


**代码功能说明：**

使用 `df -h` 和 `lsblk` 查看系统**磁盘空间**和块设备信息。

**命令原理：**

- `df -h`：显示各挂载点的磁盘使用情况（总量/已用/可用/使用率）
- `lsblk`：列出所有块设备（硬盘、分区）的树状结构

> **为什么关注磁盘？** AI 开发中，模型文件、数据集、日志可能占用大量磁盘空间。磁盘不足会导致训练中断或数据写入失败。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取磁盘信息
print("磁盘使用情况：")
!df -h

print()
print("块设备信息：")
!lsblk


### 2.1.5 CPU 与 NPU 负载监控

前面我们查看了 CPU 和 NPU 的**硬件信息**（型号、核心数等）。在实际开发中，还需要**实时监控**它们的运行负载，判断是否在满负荷运转。

- **CPU 负载**：各核心的利用率（百分比）、当前运行的进程
- **NPU 负载**：AI 核心利用率、显存使用量、温度和功率


**代码功能说明：**

使用 `top` 命令查看 CPU 的**实时利用率**和**进程列表**。

**`top` 命令原理：**

- `top` 是 Linux 下最常用的系统监控工具，实时显示 CPU、内存使用情况和进程列表
- `-b`：批处理模式（不交互，直接输出），适合在脚本/Notebook 中使用
- `-n1`：只输出 1 次快照（不是持续刷新）
- `head -20`：只看前 20 行（包含系统概览和 top 进程）

**输出中关注的内容：**

- `%Cpu(s)` 行：显示 CPU 整体利用率（us=用户态, sy=内核态, id=空闲）
- `load average`：1/5/15 分钟的平均负载（与 `uptime` 类似）

> **知识点**：CPU 利用率 = 100% - 空闲率(id)。如果 id 接近 0，说明 CPU 几乎满负荷。


In [ ]:
# 查看 CPU 实时利用率（快照）
import os
os.chdir(os.path.expanduser('~'))

print("CPU 负载监控（top 快照）：")
!top -bn1 | head -20


**代码功能说明：**

使用 `npu-smi info -t usages` 查看 NPU 的**详细资源使用情况**。

**监控指标说明：**

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">指标</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;">AICore Utilization</td>
<td style="text-align: left;">AI 核心利用率（百分比，越高说明 NPU 越忙）</td>
</tr>
<tr>
<td style="text-align: left;">Memory</td>
<td style="text-align: left;">显存使用量 / 总量</td>
</tr>
<tr>
<td style="text-align: left;">Temperature</td>
<td style="text-align: left;">芯片温度（过高会自动降频）</td>
</tr>
<tr>
<td style="text-align: left;">Power</td>
<td style="text-align: left;">当前功耗（瓦）</td>
</tr>
</table>

> **对比**：`npu-smi info` 显示概览，`npu-smi info -t usages` 显示更详细的资源使用率。类似 `nvidia-smi` 和 `nvidia-smi dmon` 的区别。


In [ ]:
# 查看 NPU 详细负载（自动解析 Card ID）
import subprocess
import os
os.chdir(os.path.expanduser('~'))

# 从 npu-smi info 解析 Card ID
result = subprocess.run(["npu-smi", "info"], capture_output=True, text=True)
if result.returncode != 0 or not result.stdout.strip():
    print("无法获取 NPU 信息，可能原因：未安装 CANN / 无 NPU 访问权限 / 未挂载 NPU")
card_ids = set()
for line in result.stdout.split("\n"):
    if not line.startswith("|"):
        continue
    fields = [f.strip() for f in line.split("|")]
    # fields[1] 是第一列：Card 行为 "<ID> <Name>"，Chip 行仅为 "<ID>"
    if len(fields) >= 2 and fields[1]:
        col1 = fields[1].split()
        if len(col1) >= 2 and col1[0].isdigit():
            card_ids.add(int(col1[0]))

if card_ids:
    for cid in sorted(card_ids):
        print(f"--- NPU Card {cid} 资源使用详情 ---")
        r = subprocess.run(["npu-smi", "info", "-t", "usages", "-i", str(cid)],
                           capture_output=True, text=True)
        if r.stdout.strip():
            print(r.stdout)
        if r.stderr.strip():
            print(f"(详细信息不可用: {r.stderr.strip()})")
else:
    print("未检测到 NPU 设备（请检查上方 npu-smi info 输出是否包含设备行）")


## 2.2 操作系统信息

获取操作系统的发行版、内核版本等信息。


**代码功能说明：**

获取**操作系统**的发行版信息和内核版本。

**命令原理：**

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">命令/文件</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;"><code>/etc/os-release</code></td>
<td style="text-align: left;">OS 发行版信息（名称、版本号、ID 等）</td>
</tr>
<tr>
<td style="text-align: left;"><code>uname -a</code></td>
<td style="text-align: left;">内核版本、主机名、架构等</td>
</tr>
<tr>
<td style="text-align: left;"><code>/proc/version</code></td>
<td style="text-align: left;">内核编译信息（GCC 版本等）</td>
</tr>
</table>

> **本环境预期**：操作系统为 Linux 的 ARM 发行版（如 Ubuntu 或 EulerOS），内核架构为 `aarch64`。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取操作系统信息
print("=" * 60)
print("操作系统信息")
print("=" * 60)

print("\n--- /etc/os-release ---")
!cat /etc/os-release

print("\n--- uname -a ---")
!uname -a

print("\n--- 内核版本 ---")
!cat /proc/version


**代码功能说明：**

查看系统**运行时间**和**负载平均值**。

**`uptime` 命令原理：**

- 显示系统已运行多长时间、当前在线用户数
- **负载平均值（Load Average）**：三个数字分别表示过去 1/5/15 分钟的平均负载
  - 负载值 = 正在运行的进程数 + 等待运行的进程数
  - 如果负载值 > CPU 核心数，说明系统过载

> **知识点**：`who -b` 显示系统上次启动时间，可用于确认是否已重启使配置生效。


In [ ]:
import os
os.chdir(os.path.expanduser('~'))  # 修复 Jupyter 工作目录问题

# 获取系统运行时间和负载
print("系统运行时间和负载平均值：")
!uptime

print("\n系统启动时间：")
!who -b


### 2.3 Python 版本信息

获取 Python 解释器的版本号和路径，确认当前使用的是 CANN 环境对应的 Python。


**代码功能说明：**

获取 **Python 解释器**的版本和路径信息。

**各属性说明：**

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">属性</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">本环境预期值</th>
</tr>
<tr>
<td style="text-align: left;"><code>sys.version</code></td>
<td style="text-align: left;">Python 完整版本号</td>
<td style="text-align: left;">3.11.4</td>
</tr>
<tr>
<td style="text-align: left;"><code>sys.executable</code></td>
<td style="text-align: left;">Python 解释器路径</td>
<td style="text-align: left;"><code>/opt/buildtools/Python-3.11.4/bin/python3.11</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sys.platform</code></td>
<td style="text-align: left;">操作系统标识</td>
<td style="text-align: left;"><code>linux</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>platform.machine()</code></td>
<td style="text-align: left;">CPU 架构</td>
<td style="text-align: left;"><code>aarch64</code>（ARM 64位）</td>
</tr>
<tr>
<td style="text-align: left;"><code>sys.byteorder</code></td>
<td style="text-align: left;">字节序</td>
<td style="text-align: left;"><code>little</code>（小端）</td>
</tr>
</table>

> **为什么关注 Python 路径？** CANN 环境的 Python 在 `/opt/buildtools/` 下，与系统默认 Python 不同。确认路径可以排查因使用错误 Python 导致的库导入失败问题。


In [ ]:
# 获取 Python 版本信息
import sys
import platform

print("=" * 60)
print("Python 环境信息")
print("=" * 60)
print(f"  Python 版本   : {sys.version}")
print(f"  Python 路径   : {sys.executable}")
print(f"  平台标识      : {sys.platform}")
print(f"  机器架构      : {platform.machine()}")
print(f"  处理器型号    : {platform.processor()}")
print(f"  操作系统      : {platform.platform()}")
print(f"  字节序        : {sys.byteorder}")


---

# 二、程序开发案例

本节实现如下案例程序，可以直接在 Jupyter 环境下运行。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">序号</th>
<th style="text-align: left;">案例</th>
<th style="text-align: left;">知识点</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">Hello World 程序</td>
<td style="text-align: left;">Python 基础语法</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">多核 CPU 并行程序</td>
<td style="text-align: left;"><code>multiprocessing</code> 多进程并行</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">CPU 与 NPU 通信</td>
<td style="text-align: left;">张量在 CPU 和 NPU 之间搬运</td>
</tr>
</table>

> 本节从最简单的程序开始，帮助初学者逐步熟悉开发环境。


## 3.1 Hello World 程序

实现一个 Python 语言实现的简单的 Hello World 程序。


**代码功能说明：**

获取 **Python 解释器**的版本和路径信息。

**各属性说明：**

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">属性</th>
<th style="text-align: left;">说明</th>
<th style="text-align: left;">本环境预期值</th>
</tr>
<tr>
<td style="text-align: left;"><code>sys.version</code></td>
<td style="text-align: left;">Python 完整版本号</td>
<td style="text-align: left;">3.11.4</td>
</tr>
<tr>
<td style="text-align: left;"><code>sys.executable</code></td>
<td style="text-align: left;">Python 解释器路径</td>
<td style="text-align: left;"><code>/opt/buildtools/Python-3.11.4/bin/python3.11</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>sys.platform</code></td>
<td style="text-align: left;">操作系统标识</td>
<td style="text-align: left;"><code>linux</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>platform.machine()</code></td>
<td style="text-align: left;">CPU 架构</td>
<td style="text-align: left;"><code>aarch64</code>（ARM 64位）</td>
</tr>
<tr>
<td style="text-align: left;"><code>sys.byteorder</code></td>
<td style="text-align: left;">字节序</td>
<td style="text-align: left;"><code>little</code>（小端）</td>
</tr>
</table>

> **为什么关注 Python 路径？** CANN 环境的 Python 在 `/opt/buildtools/` 下，与系统默认 Python 不同。确认路径可以排查因使用错误 Python 导致的库导入失败问题。


In [ ]:
# 实验3.1：Hello World 程序
# ================================

print("Hello World!")
print("Hello Ascend NPU!")
print("欢迎使用 CANN 开发环境！")

# 打印一些基本环境信息
import sys, platform
print(f"\n--- 环境信息 ---")
print(f"Python: {sys.version.split()[0]}")
print(f"平台  : {platform.machine()}")
print(f"OS    : {platform.platform()}")


## 3.2 多核 CPU 并行程序（入门）

我们的环境有 **16 个 CPU 核心**。如果任务只在一个核心上跑，就浪费了其余 15 个核心。Python 的 `multiprocessing` 模块可以让我们把任务分给多个核心**同时**执行。

### 最简单的例子：启动多个进程

下面用 4 个进程同时打印消息，感受"并行"的概念：


**代码功能说明：**

用多种方式获取 CPU 核心数，并对比结果是否一致。

**三种方式对比：**

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方式</th>
<th style="text-align: left;">原理</th>
<th style="text-align: left;">返回值</th>
</tr>
<tr>
<td style="text-align: left;"><code>os.cpu_count()</code></td>
<td style="text-align: left;">Python 标准库，读取系统信息</td>
<td style="text-align: left;">逻辑核心数</td>
</tr>
<tr>
<td style="text-align: left;"><code>multiprocessing.cpu_count()</code></td>
<td style="text-align: left;">multiprocessing 模块，用于确定并行进程数</td>
<td style="text-align: left;">逻辑核心数</td>
</tr>
<tr>
<td style="text-align: left;"><code>nproc</code></td>
<td style="text-align: left;">Shell 命令，读取 <code>/proc/cpuinfo</code></td>
<td style="text-align: left;">逻辑核心数</td>
</tr>
</table>

> 三种方式结果通常相同。如果不同，可能是因为容器或虚拟环境限制了可用核心数。


In [ ]:
# 实验3.2-A：最简单的多进程示例
import multiprocessing
import os

# 修复 Jupyter 中多进程的 getcwd 错误
os.chdir(os.path.expanduser('~'))

# 查看有多少个 CPU 核心
num_cores = os.cpu_count()
print(f"本机 CPU 核心数: {num_cores}")

# 每个工作进程要做的事情：打印一条消息
def hello_worker(worker_id):
    return f"Worker {worker_id}: Hello! (来自进程)"

# 启动 4 个进程同时执行
num = min(4, num_cores)
print(f"\n启动 {num} 个进程：")
with multiprocessing.Pool(num) as pool:
    results = pool.map(hello_worker, range(num))

for r in results:
    print(f"  {r}")
print("所有进程完成！")


### 串行 vs 并行：一个简单对比

把一个计算任务分成几份，分给多个核心同时算，看看快了多少：


**代码功能说明：**

对比**串行**（一个核心依次做）和**并行**（多个核心同时做）的执行时间，直观感受多核加速效果。

**代码逻辑：**

1. **任务**：计算 `sum(range(N))`，即 0+1+2+...+(N-1)，重复 `num` 次
2. **串行**：用 `for` 循环依次执行 `num` 次，只用 1 个 CPU 核心
3. **并行**：用 `pool.map()` 将 `num` 个任务分给 `num` 个核心**同时**执行
4. **加速比** = 串行时间 / 并行时间，衡量并行效果

**关键概念：**

- **加速比（Speedup）**：$S_p = T_{串行} / T_{并行}$，值越大越好
- **并行效率** = 加速比 / 核心数 × 100%，理想情况接近 100%
- 实际中并行效率 < 100%，因为进程创建和数据传输有额外开销

> **为什么加速比 < 核心数？** 因为启动进程、分配任务、汇总结果都需要时间，这部分开销是串行没有的。


In [ ]:
# 实验3.2-B：串行 vs 并行 简单对比
import multiprocessing
import time
import os

# 修复 Jupyter 中多进程的 getcwd 错误
os.chdir(os.path.expanduser('~'))

def simple_sum(n):
    # 计算 0+1+2+...+n-1
    return sum(range(n))

N = 5_000_000           # 计算量
num = min(4, multiprocessing.cpu_count())  # 用4个核心

# --- 串行：一个核心做4次 ---
start = time.time()
results_serial = [simple_sum(N) for _ in range(num)]
t_serial = time.time() - start

# --- 并行：4个核心各做1次，同时进行 ---
start = time.time()
with multiprocessing.Pool(num) as pool:
    results_parallel = pool.map(simple_sum, [N] * num)
t_parallel = time.time() - start

print(f"任务: 计算 0+1+2+...+{N-1}，共 {num} 次")
print(f"串行 (1核心做{num}次): {t_serial:.3f} 秒")
print(f"并行 ({num}核心各做1次): {t_parallel:.3f} 秒")
print(f"加速比: {t_serial / t_parallel:.1f}x")
print(f"结果一致: {results_serial == results_parallel}")


## 3.3 CPU 与 NPU 通信（最简单的例子）

我们的环境中有 **CPU**（通用处理器）和 **NPU**（AI 专用处理器，型号 Ascend 910B3）。它们是两个独立的处理器，各有各的内存。

**核心概念：**

- **张量（Tensor）**：PyTorch 中的数据容器，类似数组。例如 `torch.tensor([1.0, 2.0, 3.0])` 创建一个包含 3 个浮点数的张量。
- **设备（Device）**：张量存放在哪里。默认在 CPU 上，用 `.npu()` 可以搬到 NPU 上。
- **CPU 和 NPU 的内存是独立的**：NPU 不能直接读取 CPU 内存中的数据，必须先"搬"过去。

**本例演示最基本操作：在 CPU 上创建数据 → 搬到 NPU → 搬回 CPU**

> 💡 这就像你（CPU）写了一张纸条，递给同桌（NPU），同桌看完再递回来。`.npu()` 是递过去，`.cpu()` 是拿回来。


**代码功能说明：**

这段代码演示 **CPU 与 NPU 之间最基本的数据通信**，不做任何计算，只是把数据搬来搬去。

**逐行解读：**

1. `import torch` — 导入 PyTorch 深度学习框架
2. `import torch_npu` — 导入昇腾 NPU 插件（让 PyTorch 能识别 NPU 设备）
3. `torch.npu.is_available()` — 检查 NPU 是否可用
4. `x = torch.tensor([1.0, 2.0, 3.0])` — 在 CPU 上创建一个张量
5. `x.device` — 查看张量当前在哪个设备上
6. `x.npu()` — 把张量从 CPU 搬到 NPU
7. `x_npu.device` — 确认张量已在 NPU 上
8. `x_npu.cpu()` — 把张量从 NPU 搬回 CPU

> **关键理解**：`.npu()` 和 `.cpu()` 是 CPU 与 NPU 之间数据搬运的桥梁。


In [ ]:
# 实验3.3：CPU 与 NPU 之间最简单的数据通信
import os
os.chdir(os.path.expanduser('~'))

try:
    import torch
    import torch_npu

    # 检查 NPU 是否可用
    print(f"NPU 是否可用: {torch.npu.is_available()}")
    print()

    # 第1步：在 CPU 上创建一个张量（就是一组数字）
    x = torch.tensor([1.0, 2.0, 3.0])
    print(f"第1步 - CPU 上的张量: {x}")
    print(f"        所在设备: {x.device}")
    print()

    # 第2步：把数据从 CPU 搬到 NPU
    x_npu = x.npu()
    print(f"第2步 - 已搬到 NPU: {x_npu}")
    print(f"        所在设备: {x_npu.device}")
    print()

    # 第3步：把数据从 NPU 搬回 CPU
    x_back = x_npu.cpu()
    print(f"第3步 - 搬回 CPU: {x_back}")
    print(f"        所在设备: {x_back.device}")
    print()

    print("小结: 数据成功从 CPU -> NPU -> CPU 搬了一个来回！")

except ImportError as e:
    print(f"导入失败: {e}")
    print()
    print("可能原因及解决方案：")
    print("  1. torch_npu 未安装 -> 运行: pip install torch_npu")
    print("  2. 未选择正确的内核 -> 在右上角选择 'Python 3.11.4 (CANN)'")
    print("  3. CANN 环境变量未设置 -> 参考相关环境配置文档")
except Exception as e:
    print(f"运行出错: {e}")
    print("请检查 NPU 环境是否正常（运行 npu-smi info 查看）")


### 小结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">操作</th>
<th style="text-align: left;">代码</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">创建数据</td>
<td style="text-align: left;"><code>torch.tensor(...)</code></td>
<td style="text-align: left;">默认在 CPU 上创建</td>
</tr>
<tr>
<td style="text-align: left;">搬到 NPU</td>
<td style="text-align: left;"><code>.npu()</code></td>
<td style="text-align: left;">CPU → NPU（类似 GPU 的 <code>.cuda()</code>）</td>
</tr>
<tr>
<td style="text-align: left;">搬回 CPU</td>
<td style="text-align: left;"><code>.cpu()</code></td>
<td style="text-align: left;">NPU → CPU</td>
</tr>
<tr>
<td style="text-align: left;">查看设备</td>
<td style="text-align: left;"><code>.device</code></td>
<td style="text-align: left;">看张量在哪个处理器上</td>
</tr>
</table>

> 💡 **一句话记住**：`.npu()` 搬过去，`.cpu()` 搬回来。后续课程中会在 NPU 上做更多计算。


---

# 三、动手开发环节

提出几个关于**读取系统环境信息**的练习题目，便于学习者熟悉开发环境。参考答案存放在 `answer/` 目录中。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">题号</th>
<th style="text-align: left;">难度</th>
<th style="text-align: left;">题目</th>
<th style="text-align: left;">答案文件</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">⭐ 入门</td>
<td style="text-align: left;">获取并打印 CPU 型号和核心数</td>
<td style="text-align: left;"><code>answer/answer1_cpu_info.py</code></td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">⭐ 入门</td>
<td style="text-align: left;">获取并打印 NPU 型号和健康状态</td>
<td style="text-align: left;"><code>answer/answer2_npu_info.py</code></td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">⭐⭐ 基硷</td>
<td style="text-align: left;">获取内存总量与使用率，格式化输出</td>
<td style="text-align: left;"><code>answer/answer3_memory_info.py</code></td>
</tr>
</table>

> 💡 **建议**：先独立思考并尝试编写代码，遇到困难时再参考 `answer/` 目录中的答案。


## 题目 1（⭐ 入门）：获取并打印 CPU 型号和核心数

**要求**：编写 1-2 行命令或 Python 代码，获取并打印：
- CPU 架构（如 aarch64）
- CPU 型号名称
- CPU 核心数

**提示**：
- 可使用 `!lscpu` 命令
- 或使用 Python `os.cpu_count()` 和 `platform` 模块


## 题目 2（⭐ 入门）：获取并打印 NPU 型号和健康状态

**要求**：编写 1-2 行命令，获取并打印：
- NPU 型号（如 Ascend 910B3）
- NPU 健康状态（如 OK）
- NPU 功率和温度

**提示**：
- 使用 `!npu-smi info` 命令
- 观察输出中的 Name、Health、Power、Temp 等字段


## 题目 3（⭐⭐ 基硎）：获取内存总量与使用率

**要求**：编写 Python 代码，获取并格式化输出：
- 内存总量（GB）
- 已使用内存（GB）
- 内存使用率（%）
- 交换分区信息

**提示**：
- 使用 `!free -h` 或 `!cat /proc/meminfo` 命令
- 或使用 Python 读取 `/proc/meminfo` 并解析


**代码功能说明：**

这是学生的**动手练习区域**。请在下方编写代码，尝试完成第四节的练习题目。

> **提示**：可以参考前面章节的代码示例，尝试使用 `!命令` 运行 Shell 命令，或使用 Python 标准库获取系统信息。


In [ ]:
# ==================== 学生动手练习区域 ====================
# 请在此处编写你的代码

# 例如：题目1 - 格式化输出系统信息摘要
# import subprocess, os, platform, sys
# ...

print("请在此处编写你的代码...")


---

# 实验总结

通过本实验，你应该掌握了以下内容：

### 环境信息获取
- [x] CPU 信息：`lscpu`、`/proc/cpuinfo`、`os.cpu_count()`
- [x] NPU 信息：`npu-smi info`
- [x] 内存信息：`free -h`、`/proc/meminfo`
- [x] OS 信息：`/etc/os-release`、`uname -a`
- [x] Python 信息：`sys.version`

### 程序开发
- [x] Python 基础程序（Hello World）
- [x] 多核 CPU 并行程序（`multiprocessing`）
- [x] CPU 与 NPU 数据通信（`.npu()` / `.cpu()`）

### 关键概念

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">概念</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">NPU</td>
<td style="text-align: left;">Neural Processing Unit，神经网络专用处理器</td>
</tr>
<tr>
<td style="text-align: left;">Ascend 910B3</td>
<td style="text-align: left;">本实验使用的昇腾 AI 处理器型号</td>
</tr>
<tr>
<td style="text-align: left;"><code>npu-smi</code></td>
<td style="text-align: left;">昇腾 NPU 系统管理工具（类似 nvidia-smi）</td>
</tr>
<tr>
<td style="text-align: left;"><code>multiprocessing</code></td>
<td style="text-align: left;">Python 多进程模块，利用多核 CPU 并行计算</td>
</tr>
<tr>
<td style="text-align: left;"><code>.npu()</code></td>
<td style="text-align: left;">把数据从 CPU 搬到 NPU</td>
</tr>
<tr>
<td style="text-align: left;"><code>.cpu()</code></td>
<td style="text-align: left;">把数据从 NPU 搬回 CPU</td>
</tr>
</table>

---

> 📁 参考答案位于 `answer/` 目录中  
